In [1]:
import compressai
compressai

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/conda/lib/python3.10/site-packages/compressai/models/video/google.py:353: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)


<module 'compressai' from '/opt/conda/lib/python3.10/site-packages/compressai/__init__.py'>

In [18]:
import torch
import time
import numpy as np
from PIL import Image
import requests
from transformers import CLIPConfig, CLIPModel, CLIPProcessor


# model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)


In [21]:
device = "cuda:7"

model.eval()
model.to(device)
inputs.to(device)
BATCH_SIZE = inputs['pixel_values'].shape[0]
print(f"Device: {device}")
print(f"Batch Size: {inputs['pixel_values'].shape[0]}")

# ---------------------------------------------------------
# 3. 측정 로직 (Warm-up -> Loop -> Sync)
# ---------------------------------------------------------

# (A) Warm-up: 초기화 오버헤드 제거 (GPU 예열)
print("Warming up...")
with torch.inference_mode():
    for _ in range(50):
        _ = model.get_image_features(pixel_values=inputs['pixel_values'])
        
torch.cuda.synchronize() # 웜업 끝날 때까지 대기

# (B) 실제 측정: torch.cuda.Event 사용 (가장 정확함)
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
repetitions = 100  # 반복 횟수 (많을수록 평균이 정확함)
timings = []

print("Measuring...")
with torch.inference_mode():
    for _ in range(repetitions):
        start_event.record()
        _ = model.get_image_features(pixel_values=inputs['pixel_values'])
        end_event.record()
        
        # 이 루프 안에서는 동기화를 하지 않아야 GPU가 풀로 돕니다.
        # 시간 기록만 해두고 나중에 계산합니다.
        end_event.synchronize() 
        curr_time = start_event.elapsed_time(end_event) # 밀리초(ms) 반환
        timings.append(curr_time)

# ---------------------------------------------------------
# 4. 결과 계산 (Latency & Throughput)
# ---------------------------------------------------------
timings = np.array(timings)
mean_latency_ms = np.mean(timings)
std_latency_ms = np.std(timings)
throughput = (BATCH_SIZE * 1000) / mean_latency_ms # images/sec

print("-" * 30)
print(f"Mean Latency: {mean_latency_ms:.4f} ms per batch")
print(f"Std Latency : {std_latency_ms:.4f} ms")
print(f"Throughput  : {throughput:.2f} samples/sec")
print("-" * 30)

Device: cuda:7
Batch Size: 1
Warming up...
Measuring...
------------------------------
Mean Latency: 2.8610 ms per batch
Std Latency : 0.9306 ms
Throughput  : 349.53 samples/sec
------------------------------


Device: cuda:7
Batch Size: 1
Warming up...
Measuring...
------------------------------
Mean Latency: 2.5179 ms per batch
Std Latency : 0.5389 ms
Throughput  : 397.15 samples/sec
------------------------------

In [22]:
import torch
import numpy as np
import time

# 설정
device = "cuda:7"
TARGET_BATCH_SIZE = 128  # 목표 배치 사이즈 설정

# ---------------------------------------------------------
# 0. 입력 데이터 배치 조정 (Batch Size 128로 만들기)
# ---------------------------------------------------------
# 기존 inputs['pixel_values']의 첫 번째 샘플을 복제하여 128개로 만듭니다.
# (Shape 예시: [1, 3, 224, 224] -> [128, 3, 224, 224])
if inputs['pixel_values'].ndim == 4:
    sample_image = inputs['pixel_values'][0:1] # 첫 한 장만 추출
    # 메모리 효율을 위해 .expand() 대신 실제 메모리를 할당하는 .repeat() 사용 권장 (Inference 측정 시)
    batch_pixel_values = sample_image.repeat(TARGET_BATCH_SIZE, 1, 1, 1)
    inputs['pixel_values'] = batch_pixel_values

# ---------------------------------------------------------
# 1. 모델 및 데이터 준비
# ---------------------------------------------------------
model.eval()
model.to(device)
inputs = inputs.to(device) # 데이터를 GPU로 이동

BATCH_SIZE = inputs['pixel_values'].shape[0]

print(f"Device: {device}")
print(f"Current Batch Size: {BATCH_SIZE}") # 128이 나와야 함

# ---------------------------------------------------------
# 2. 측정 로직 (Warm-up -> Loop -> Sync)
# ---------------------------------------------------------

# (A) Warm-up: 초기화 오버헤드 제거 (GPU 예열)
print("Warming up...")
with torch.inference_mode():
    for _ in range(10): # 배치 사이즈가 커졌으므로 웜업 횟수는 조금 줄여도 됨 (10~50회)
        _ = model.get_image_features(pixel_values=inputs['pixel_values'])
        
torch.cuda.synchronize() # 웜업 끝날 때까지 대기

# (B) 실제 측정
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
repetitions = 100 
timings = []

print(f"Measuring latency with batch size {BATCH_SIZE}...")

with torch.inference_mode():
    for _ in range(repetitions):
        start_event.record()
        _ = model.get_image_features(pixel_values=inputs['pixel_values'])
        end_event.record()
        
        # 정확한 커널 실행 시간 측정을 위해 각 루프마다 동기화
        end_event.synchronize() 
        curr_time = start_event.elapsed_time(end_event) # ms 반환
        timings.append(curr_time)

# ---------------------------------------------------------
# 3. 결과 계산 (Latency & Throughput)
# ---------------------------------------------------------
timings = np.array(timings)
mean_latency_ms = np.mean(timings)
std_latency_ms = np.std(timings)
# Throughput = (Batch Size * 1000ms) / Latency(ms)
throughput = (BATCH_SIZE * 1000) / mean_latency_ms 

print("-" * 30)
print(f"Batch Size  : {BATCH_SIZE}")
print(f"Mean Latency: {mean_latency_ms:.4f} ms")
print(f"Std Latency : {std_latency_ms:.4f} ms")
print(f"Throughput  : {throughput:.2f} images/sec")
print("-" * 30)

Device: cuda:7
Current Batch Size: 128
Warming up...
Measuring latency with batch size 128...
------------------------------
Batch Size  : 128
Mean Latency: 317.9660 ms
Std Latency : 1.8252 ms
Throughput  : 402.56 images/sec
------------------------------


Device: cuda:7
Current Batch Size: 128
Warming up...
Measuring latency with batch size 128...
------------------------------
Batch Size  : 128
Mean Latency: 317.8404 ms
Std Latency : 1.9231 ms
Throughput  : 402.72 images/sec
------------------------------

In [ ]:
import torch
import time
import numpy as np # 평균, 표준편차 계산용
from transformers import CLIPModel

model_id = "openai/clip-vit-base-patch16"

print(f"Loading model '{model_id}' to CPU RAM first...")
full_model_cpu = CLIPModel.from_pretrained(model_id)
partial_model_cpu = CLIPModel.from_pretrained(model_id)

vision_part_cpu = partial_model_cpu.vision_model

print("Ready to measure transfer time (CPU -> GPU).\n")

device = 'cuda:7' 

# -------------------------------------------------------
# 수정된 측정 함수 (100회 반복)
# -------------------------------------------------------
def measure_transfer_time_stats(module, name, repetitions=100):
    timings = []
    
    # Warm-up (첫 실행 오버헤드 제거를 위해 1회 미리 실행)
    module.to(device)
    torch.cuda.synchronize()
    module.cpu() # 다시 CPU로 복구
    torch.cuda.synchronize()
    
    print(f"Measuring [{name}] over {repetitions} runs...")
    
    for _ in range(repetitions):
        # 1. 측정 전 준비: 모델이 확실히 CPU에 있는지 확인
        # (루프 끝에서 복구하지만, 안전을 위해 보장)
        
        # 2. 캐시 비우기 & 동기화 (순수 전송 시간 측정을 위함)
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
        # 3. 시간 측정 시작
        start_time = time.perf_counter()
        
        module.to(device) # CPU -> GPU 전송
        
        torch.cuda.synchronize() # 전송 완료 대기
        end_time = time.perf_counter()
        
        # 4. 기록 (ms 단위)
        timings.append((end_time - start_time) * 1000)
        
        # 5. [중요] 다음 측정을 위해 다시 CPU로 내리기 (Reset)
        # 이 시간은 측정에 포함되지 않습니다.
        module.cpu()
        torch.cuda.synchronize()

    # 결과 계산
    timings = np.array(timings)
    mean_ms = np.mean(timings)
    std_ms = np.std(timings)
     
    print(f"   - Mean Transfer Time : {mean_ms:.4f} ms")
    print(f"   - Std Deviation      : {std_ms:.4f} ms")
    print(f"   - Min Transfer Time  : {np.min(timings):.4f} ms") # 이게 진짜 성능
    print(f"   - Max Transfer Time  : {np.max(timings):.4f} ms") # 이게 졸다가 깬 경우
    param_count = sum(p.numel() for p in module.parameters())
    print(f"   - Parameters         : {param_count:,}")
    print("-" * 50)

# -------------------------------------------------------
# 실제 측정 수행
# -------------------------------------------------------

# Case 1: 모델 전체 이동
measure_transfer_time_stats(full_model_cpu, "Full CLIP Model", rßepetitions=1000)

# Case 2: 이미지 인코딩 부분만 이동
# measure_transfer_time_stats(vision_part_cpu, "Vision Part Only", repetitions=1000)

Loading model 'openai/clip-vit-base-patch16' to CPU RAM first...
Ready to measure transfer time (CPU -> GPU).

Measuring [Full CLIP Model] over 1000 runs...
   - Mean Transfer Time : 127.5993 ms
   - Std Deviation      : 66.4415 ms
   - Parameters         : 149,620,737
--------------------------------------------------


In [56]:
device = 'cuda:7'
tensor_data = torch.randn(4096, 4096, dtype=torch.float32)

print(f"Tensor created. Shape: {tensor_data.shape}")
print(f"Memory Size: {tensor_data.nelement() * tensor_data.element_size() / 1024 / 1024:.2f} MB")

# -------------------------------------------------------
# 텐서 전용 측정 함수
# -------------------------------------------------------
def measure_tensor_transfer(tensor, name, repetitions=100):
    timings = []
    
    # Warm-up
    tensor = tensor.to(device)
    torch.cuda.synchronize()
    tensor = tensor.cpu()
    torch.cuda.synchronize()
    
    print(f"Measuring [{name}] over {repetitions} runs...")
    
    for _ in range(repetitions):
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
        start_time = time.perf_counter()
        
        # [핵심] 텐서 이동
        # non_blocking=True를 쓰면 비동기 전송이 되지만, 
        # 정확한 시간 측정을 위해 여기서는 기본값(False) 또는 sync를 사용합니다.
        target = tensor.to(device) 
        
        torch.cuda.synchronize()
        end_time = time.perf_counter()
        
        timings.append((end_time - start_time) * 1000)
        
        # Reset (다시 CPU로)
        del target # GPU 메모리 해제
        # 원본 tensor는 이미 CPU에 있으므로 별도 조치 불필요
        # (단, to()는 복사본을 만들기 때문에 원본은 CPU에 그대로 있습니다)

    timings = np.array(timings)
    print(f"[{name}] Results:")
    print(f"   - Mean Transfer Time : {np.mean(timings):.4f} ms")
    print(f"   - Min Transfer Time  : {np.min(timings):.4f} ms") # 가장 빠른 속도
    print(f"   - Max Transfer Time  : {np.max(timings):.4f} ms")
    print(f"   - Std Deviation      : {np.std(timings):.4f} ms")
    print("-" * 50)

# 실행
measure_tensor_transfer(tensor_data, "4096 x 4096 Tensor", repetitions=1000)

Tensor created. Shape: torch.Size([4096, 4096])
Memory Size: 64.00 MB
Measuring [4096 x 4096 Tensor] over 1000 runs...
[4096 x 4096 Tensor] Results:
   - Mean Transfer Time : 6.9268 ms
   - Min Transfer Time  : 4.5966 ms
   - Max Transfer Time  : 43.7418 ms
   - Std Deviation      : 2.9417 ms
--------------------------------------------------
